# EDA: 경구약제 객체 탐지 (pegstar, 모델 A 관점)

원본 라벨(`data/raw/train_annotations`)을 직접 읽어 분석한다. `class_map.json` 없이도 돌아간다.

- 커널: `.venv` 선택 후 **Run All**
- 대회 이미지가 들어간 결과(박스 그림)는 **디스코드로만** 공유, PR·이슈에는 숫자만
- 노트북 출력은 커밋 시 nbstripout이 자동으로 지운다

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw

# 한글 폰트 (Windows: 맑은 고딕)
if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
ANN = RAW / "train_annotations"
IMG = RAW / "train_images"
TEST = RAW / "test_images"
print("ROOT:", ROOT)
print("JSON 파일 수:", sum(1 for _ in ANN.rglob("*.json")))

## 1. 라벨 읽기
알약 1개당 JSON 1개라서, 모든 JSON을 모은 뒤 같은 박스는 한 번만 남긴다. bbox는 COCO `[x, y, w, h]` (원본 픽셀).

In [ ]:
META = [
    "drug_shape",
    "color_class1",
    "print_front",
    "print_back",
    "back_color",
    "light_color",
    "drug_dir",
    "camera_la",
    "camera_lo",
    "dl_name",
]

rows = []
for jp in sorted(ANN.rglob("*.json")):
    d = json.loads(jp.read_text(encoding="utf-8"))
    imgs = {im["id"]: im for im in d.get("images", [])}
    cats = {c["id"]: c.get("name", str(c["id"])) for c in d.get("categories", [])}
    for a in d.get("annotations", []):
        im = imgs.get(a["image_id"], {})
        x, y, w, h = a["bbox"]
        row = {
            "file_name": im.get("file_name"),
            "img_w": im.get("width"),
            "img_h": im.get("height"),
            "category_id": a["category_id"],
            "name": cats.get(a["category_id"]),
            "x": x,
            "y": y,
            "w": w,
            "h": h,
            "json": jp.relative_to(ANN).as_posix(),
        }
        row.update({k: im.get(k) for k in META})
        rows.append(row)

raw = pd.DataFrame(rows)
df = raw.drop_duplicates(["file_name", "category_id", "x", "y", "w", "h"]).reset_index(drop=True)
print(f"읽은 박스 {len(raw)}개 → 중복 제거 후 {len(df)}개")
df.head()

## 2. 기본 숫자

In [ ]:
img_files = sorted(p.name for p in IMG.glob("*.png"))
per_img = df.groupby("file_name").size()

print("학습 이미지 (폴더):", len(img_files))
print("라벨이 있는 이미지:", df.file_name.nunique())
print("클래스(약) 수:", df.category_id.nunique())
print("총 박스 수:", len(df))
print("\n이미지 크기 (w × h):")
print(df.drop_duplicates("file_name")[["img_w", "img_h"]].value_counts())
print("\n이미지당 알약(박스) 수:")
print(per_img.value_counts().sort_index())

## 3. 클래스 분포
클래스마다 이미지가 몇 장 있는지. 적은 클래스가 많으면 과적합 위험이 크고, 큰 모델(s → m)은 신중해야 한다.

In [ ]:
per_cls = (
    df.groupby(["category_id", "name"])
    .file_name.nunique()
    .rename("images")
    .sort_values()
    .reset_index()
)
print(per_cls.images.describe().round(1))
print("\n이미지 1~2장뿐인 클래스:", (per_cls.images <= 2).sum(), "개")
print("\n가장 적은 10개:")
print(per_cls.head(10).to_string(index=False))
print("\n가장 많은 10개:")
print(per_cls.tail(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(range(len(per_cls)), per_cls.images)
ax.set_xlabel("클래스 (이미지 수 순)")
ax.set_ylabel("이미지 수")
ax.set_title("클래스별 학습 이미지 수")
plt.tight_layout()

## 4. 박스 크기 (모델 A 핵심)
`imgsz`로 줄였을 때 박스의 짧은 변이 몇 픽셀이 되는지 본다. 충분히 크면 640으로도 되고, 작으면 1024 실험의 근거가 된다. 대회 지표가 IoU 0.75 이상만 인정하므로 박스 경계 정밀도가 중요하다.

In [ ]:
long_side = df[["img_w", "img_h"]].max(axis=1)
short_box = df[["w", "h"]].min(axis=1)
df["rel_w"] = df.w / df.img_w
df["rel_h"] = df.h / df.img_h
df["ratio"] = df.w / df.h
df["area_frac"] = df.w * df.h / (df.img_w * df.img_h)
df["short_640"] = short_box * 640 / long_side
df["short_1024"] = short_box * 1024 / long_side

print(
    df[["w", "h", "rel_w", "rel_h", "ratio", "area_frac", "short_640", "short_1024"]]
    .describe(percentiles=[0.05, 0.5, 0.95])
    .round(3)
)
print("\n640에서 짧은 변 32px 미만 박스:", (df.short_640 < 32).sum(), "개")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
ax[0].hist(df.rel_w, bins=30)
ax[0].set_title("박스 너비 / 이미지 너비")
ax[1].hist(df.ratio, bins=30)
ax[1].set_title("박스 가로세로비 (w/h)")
ax[2].hist(df.short_640, bins=30, alpha=0.7, label="640")
ax[2].hist(df.short_1024, bins=30, alpha=0.7, label="1024")
ax[2].set_title("축소 후 박스 짧은 변 (px)")
ax[2].legend()
plt.tight_layout()

In [ ]:
# 박스 중심 위치 (이미지 대비 0~1)
cx = (df.x + df.w / 2) / df.img_w
cy = (df.y + df.h / 2) / df.img_h
fig, ax = plt.subplots(figsize=(4, 5))
ax.scatter(cx, cy, s=8, alpha=0.5)
ax.set_xlim(0, 1)
ax.set_ylim(1, 0)
ax.set_title("박스 중심 위치")
plt.tight_layout()

## 5. 촬영 조건과 알약 속성
조명·배경·촬영 각도가 다양한지 보고, 증강(색·회전) 실험의 근거로 쓴다.

In [ ]:
imgs = df.drop_duplicates("file_name")
for col in ["back_color", "light_color", "drug_dir", "camera_la", "camera_lo"]:
    print(f"[{col}]")
    print(imgs[col].value_counts(dropna=False).to_string(), "\n")

cls_attr = df.drop_duplicates("category_id")
print("[모양]")
print(cls_attr.drug_shape.value_counts().to_string(), "\n")
print("[색]")
print(cls_attr.color_class1.value_counts().to_string())

## 6. 헷갈릴 만한 약 (hard negative 후보)
모양과 색이 같은 클래스끼리 묶었다. 각인(`print_front`)으로만 구분되는 약들이라 오분류 분석 때 먼저 볼 후보다.

In [ ]:
grp = (
    cls_attr.groupby(["drug_shape", "color_class1"])
    .agg(n=("category_id", "size"), names=("name", list), prints=("print_front", list))
    .query("n > 1")
    .sort_values("n", ascending=False)
)
print("모양·색이 같은 그룹:", len(grp), "개")
for (shape, color), r in grp.iterrows():
    print(f"\n{shape} / {color} ({r.n}종)")
    for nm, pr in zip(r.names, r.prints):
        print(f"   - {nm}  [각인: {pr}]")

## 7. 라벨 품질
파일 이름 앞부분(`K-001900-016548-...`)에 들어 있는 약 코드 수 = 사진 속 알약 수로 보고, 라벨 박스 수와 비교한다. 적으면 **라벨 누락** 의심.

In [ ]:
def n_codes(fname):
    return len(fname.split("_")[0].split("-")) - 1


chk = per_img.rename("boxes").reset_index()
chk["codes_in_name"] = chk.file_name.map(n_codes)
chk["missing"] = chk.codes_in_name - chk.boxes

bad = df[
    (df.w <= 0)
    | (df.h <= 0)
    | (df.x < 0)
    | (df.y < 0)
    | (df.x + df.w > df.img_w)
    | (df.y + df.h > df.img_h)
]
no_label = sorted(set(img_files) - set(df.file_name))
dup_cls = df.groupby(["file_name", "category_id"]).size().loc[lambda s: s > 1]

print("이미지 밖으로 나가거나 크기가 0인 박스:", len(bad))
print("라벨이 하나도 없는 이미지:", len(no_label))
print("박스 4개 초과 이미지:", (per_img > 4).sum())
print("같은 이미지에 같은 클래스 박스가 2개 이상:", len(dup_cls))
print("\n이름 속 알약 수 vs 박스 수:")
print(chk.missing.value_counts().sort_index().rename("이미지 수").to_string())
print("\n박스가 부족한 이미지 (최대 10개):")
print(chk[chk.missing > 0].head(10).to_string(index=False))

## 8. 테스트 이미지와 비교
학습과 테스트의 이미지 크기가 같은지 확인한다 (파일 헤더만 읽어 빠르다).

In [ ]:
test_sizes = pd.Series([Image.open(p).size for p in TEST.glob("*.png")]).value_counts()
print("테스트 이미지:", int(test_sizes.sum()), "장")
print(test_sizes.rename("장수").to_string())

## 9. 박스 그려 보기 (로컬 확인용, 공개 금지)

In [ ]:
def show(fname, scale=3):
    im = Image.open(IMG / fname).convert("RGB")
    dr = ImageDraw.Draw(im)
    for r in df[df.file_name == fname].itertuples():
        dr.rectangle([r.x, r.y, r.x + r.w, r.y + r.h], outline="red", width=6)
    print(fname)
    print(df[df.file_name == fname][["name", "x", "y", "w", "h"]].to_string(index=False))
    return im.resize((im.width // scale, im.height // scale))


show(df.file_name.iloc[0])

In [ ]:
# 박스가 부족한 이미지가 있으면 직접 확인
missing = [f for f in chk[chk.missing > 0].file_name if (IMG / f).exists()]
show(missing[0]) if missing else print("박스가 부족한 이미지 없음")

## 10. 디스코드 공유용 요약

In [ ]:
summary = f"""[EDA 요약] pegstar (모델 A 관점)
- 학습 이미지 {len(img_files)}장 (라벨 있음 {df.file_name.nunique()}장), 클래스 {df.category_id.nunique()}개, 박스 {len(df)}개
- 이미지당 알약 {per_img.min()}~{per_img.max()}개 (평균 {per_img.mean():.2f})
- 클래스별 이미지 수 중앙값 {per_cls.images.median():.0f}장, 1~2장뿐인 클래스 {(per_cls.images <= 2).sum()}개
- 640 기준 박스 짧은 변 중앙값 {df.short_640.median():.0f}px (최소 {df.short_640.min():.0f}px)
- 모양·색이 같은 약 그룹 {len(grp)}개 (각인으로만 구분)
- 라벨 의심: 범위 밖 박스 {len(bad)}개, 라벨 없는 이미지 {len(no_label)}장, 박스가 이름보다 적은 이미지 {(chk.missing > 0).sum()}장
"""
print(summary)